# **Building GPT From Scratch**

In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-01-14 13:39:32--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-01-14 13:39:32 (35.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



**Reading The Text**






In [2]:
with open("input.txt", "r" , encoding="utf-8") as f:
  text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


# **Encode the entire text dataset**

In [7]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # the 100 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


**Splitting Train and Test**

In [8]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [11]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [12]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


# **Bigram Language Model (Baseline)**
This code defines a Bigram Model, the simplest possible language model. It predicts the next character based only on the current character, ignoring all other history. It serves as a baseline to compare our Transformer against later.

**1. The Setup & Initialization (__init__)**
Inheritance: nn.Module is the standard PyTorch base class. It handles weight tracking and gradient calculation automatically.

The "Brain": token_embedding_table is the only layer in this model.

It is a lookup table of size (vocab_size, vocab_size).

Logic: For every input character (row), it learns a score (logit) for every possible next character (column).

Note: In a real Transformer, the second dimension would be n_embd (e.g., 64 or 384), but here we output the logits directly.

**2. The Forward Pass (forward)**
This function computes the predictions (logits) and, if provided, the error (loss).

Inputs:

idx: The input integers of shape (Batch, Time).

targets: The correct expected next integers (optional).

The Lookup: logits = self.token_embedding_table(idx)

This converts input indices into raw scores. Shape becomes (Batch, Time, Channels).

The Loss Calculation (Crucial Part):

Why reshape? PyTorch's F.cross_entropy expects inputs in the shape (N, C) (where N is total examples, C is classes). It does not like multidimensional (B, T, C) inputs natively.

logits.view(B*T, C): We squash the Batch and Time dimensions together. Instead of "4 batches of 8 characters," we treat it as "32 independent characters."

F.cross_entropy: Calculates how wrong the predictions were compared to the targets.

**3. The Generation Loop (generate)**
This function creates new text. It is used after training.

The Input: idx keeps growing. It starts as one character, then becomes 2, then 3...

The Step-by-Step Logic:

self(idx): Feed the entire history into the model to get logits for every step.

logits[:, -1, :]: Important. We only care about the prediction for the last character. We throw away the predictions for the previous history.

F.softmax: Converts raw scores (e.g., 5.0, -1.0) into probabilities (e.g., 0.95, 0.05).

torch.multinomial: This samples the next character randomly based on the probabilities.

Why not argmax? If we always picked the highest probability, the model would be repetitive and boring. Sampling introduces variety.

torch.cat: Appends the new character to the sequence so it can be used as context for the next loop.

**💡 Visual Reference for the Reshaping**
When calculating loss, we flatten the tensors:

Batch (B): 4 sequences

Time (T): 8 characters each

Channels (C): 65 possible characters

Result: A long list of 4 * 8 = 32 independent classification problems.

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [14]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

**This block implements Stochastic Gradient Descent (SGD). It runs thousands of times, slightly tweaking the model's parameters in every iteration to minimize the error (loss).**

# **1. Hyperparameters**
batch_size = 32: We increased this from 4 to 32.

Why? A larger batch provides a more stable estimate of the gradient. Averaging the error over 32 examples reduces noise and prevents the model from reacting too wildly to a single "weird" data point.

range(10000): The number of training iterations.

Scale: 100 steps is barely enough to verify the code runs. For a small model like this, 5,000–10,000 steps are needed to see the loss drop significantly (e.g., from ~4.2 down to ~2.0).

# **2. The Data Fetch (get_batch)**
xb, yb = get_batch('train')

In every step, we grab a random chunk of data from the training set.

The model never sees the entire dataset at once; it learns from these small, random "mini-batches."

# **3. The Forward Pass**
logits, loss = m(xb, yb)

We feed the inputs xb into the model.

The model makes a guess and compares it to the targets yb to calculate the loss (Negative Log Likelihood).

# **4. The Backward Pass (The "Holy Trinity" of PyTorch)**
These three lines are the core of neural network training:

optimizer.zero_grad(set_to_none=True):

Reset: We must clear the gradients from the previous step. If we don't, PyTorch will accumulate (add) the new gradients to the old ones, causing the updates to explode.

Note: set_to_none=True is slightly more efficient than setting them to 0.

loss.backward():

Backpropagation: This calculates the gradient for every parameter. It answers: "In which direction should I nudge this weight to lower the loss?"

optimizer.step():

Update: The optimizer takes the calculated gradients and updates the model's weights using the Learning Rate (e.g., weight = weight - learning_rate * gradient).

# 5. Monitoring
print(loss.item()): extracting the float value from the tensor to track progress. We want this number to go down over time!

In [15]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


print(loss.item())


2.382369041442871


In [16]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulsee


In [17]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [18]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])


---

### 📝 Notes: Version 1 - Manual Context Aggregation (The Loop Approach)

This block demonstrates the simplest way to let a token "see" its past: by calculating the average of all previous tokens using standard Python loops.

#### 1. The Goal: "Bag of Words" (`xbow`)

* **The Problem:** Up to this point, tokens were independent. The token at index 5 didn't know what happened at index 4.
* **The Solution:** We create a new tensor `xbow` (Bag of Words) where the vector at time `t` is the **average** of all vectors from time `0` up to `t`.
* **Mathematical Notation:**

#### 2. The Logic Breakdown

* **`xbow = torch.zeros((B,T,C))`**:
* We start with an empty container of the same shape as our input.


* **The Nested Loops:**
* `for b in range(B)`: Iterate through every sequence in the batch (independent examples).
* `for t in range(T)`: Iterate through every time step (character) in the sequence.


* **The Context Slice (Crucial):**
* `xprev = x[b, :t+1]`
* **":t+1"**: This is the "Look Back." If we are at step `t=2`, this slice grabs indices `0, 1, 2`. It captures the **Past + Present**.


* **The Aggregation:**
* `xbow[b,t] = torch.mean(xprev, 0)`
* We squash the previous tokens into a single average vector. This summarizes the context "bag."



#### 3. Why is this important?

This is the conceptual precursor to **Self-Attention**.

* In this version, the "Attention" is uniform (average). Every past word is equally important.
* In the final Transformer, we will replace this "dumb average" with a "weighted average" where the model decides which past words are important.

#### 4. Performance Note (Why we replace this later)

* **Bottleneck:** Using nested Python `for` loops is extremely slow on GPUs.
* **Next Step:** We will replace this entire block with **Matrix Multiplication** (`wei @ x`), which performs the exact same calculation instantly using Linear Algebra.

---

In [19]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)



### 📝 Notes: Version 2 - Weighted Aggregation (The Matrix Trick)

This block implements the exact same logic as Version 1 (averaging past tokens), but it uses **Matrix Multiplication** to do it all at once. This is the efficient "vectorized" way that GPUs love.

#### 1. The Weight Matrix (`wei`)

We construct a matrix that acts as a "filter" or "recipe" for how to combine tokens.

* **`wei = torch.tril(torch.ones(T, T))`**:
* Creates a **Lower Triangular** matrix of 1s.
* **The "Mask":** The zeros in the upper-right corner represent the future. Token 0 cannot see Token 1. Token 2 cannot see Token 3.
* **Visual:**



* **`wei = wei / wei.sum(1, keepdim=True)`**:
* **Normalization:** We divide each row by the sum of that row (e.g., Row 2 sums to 3, so we divide by 3).
* **Result:** The rows now sum to **1.0**. This turns the "sum" into an "average."
* **New Visual:**




#### 2. The Matrix Multiplication (`wei @ x`)

This is the most important line to understand for Transformers.

* **The Operation:** `(T, T) @ (B, T, C)`.
* **Broadcasting:** PyTorch realizes `wei` is missing the Batch dimension `(B)`, so it applies the same `wei` matrix to every single batch item in parallel.
* **The Math:**
* For the 3rd token (Row 2 of `wei`), the dot product is:


* This is mathematically identical to taking the mean.



#### 3. Verification (`torch.allclose`)

* **`torch.allclose(xbow, xbow2)`**:
* Computers use floating-point numbers (e.g., 0.333333...), so values are rarely *exactly* equal.
* `allclose` checks if the numbers are "close enough" (within a tiny margin of error).
* **Result:** `True`. This proves our fast matrix math does the exact same job as the slow loops.



---

In [20]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False



### 📝 Notes: Version 3 - Softmax & Masking (The Standard Approach)

This block implements the **exact mechanism** used in modern Transformers (like GPT-4) to aggregate information. While the output is still just a simple average (because we started with zeros), the **method**—using Masking and Softmax—is what allows the model to learn.

#### 1. The Setup (`tril` & `zeros`)

* **`tril`**: The "Lower Triangular" matrix. It defines the rules of the game: you can only see the past.
* **`wei = torch.zeros((T, T))`**:
* Here, we initialize the "affinity scores" to 0.
* **Meaning:** "Currently, every token has equal 0 preference for every other token."
* *Note:* In the next version (Self-Attention), these zeros will be replaced by calculated scores (`Q @ K`).



#### 2. The Masking (`masked_fill`)

This is the standard way to implement "Autoregressive" behavior (preventing the model from seeing the future).

* **Code:** `wei = wei.masked_fill(tril == 0, float('-inf'))`
* **Logic:** Wherever the `tril` is 0 (the future positions), we force the weight to be **Negative Infinity** ().
* **Visual:**


#### 3. The Softmax (`F.softmax`)

This is the normalization layer. It converts raw scores (logits) into probabilities.

* **How Softmax handles `-inf`:**
* The formula involves .
* .
* **Result:** The probability of attending to a future token becomes **zero**.


* **How Softmax handles `0`:**
* .
* If a row has three valid past tokens (three `0`s), they all get a score of 1. Softmax then divides them by the sum (3).
* **Result:** `[0.33, 0.33, 0.33]`.



#### 4. Why use Softmax instead of simple division?

In Version 2, we just divided by the sum. Why do this complex exponent math?

* **Differentiability:** Softmax plays very nicely with Backpropagation.
* **Selectivity:** Later, when our scores aren't just zeros but real numbers (e.g., `25, -10, 5`), Softmax will naturally make the high scores huge (near 1.0) and the low scores tiny (near 0.0). It acts like a "contrast knob," allowing the model to focus intensely on specific words.

---

In [21]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False


---

### 📝 Notes: Version 4 - Self-Attention (The "Search" Engine)

This block represents the leap from "dumb averaging" to "intelligent routing." Instead of mixing past tokens equally (or by proximity), the model now **searches** for relevant information from the past based on content.

#### 1. The Core Idea: Q, K, V

In previous versions, we used the token's position to decide what to aggregate. Now, we emit three vectors for every token to decide based on **meaning**.

* **Query (`q`)**: "What am I looking for?" (e.g., *I am a verb looking for my noun subject*).
* **Key (`k`)**: "What do I contain?" (e.g., *I am a noun, specifically 'Cat'*).
* **Value (`v`)**: "If you find me interesting, here is the information I will give you." (e.g., *'Cat' attributes*).

#### 2. The Linear Projections

```python
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

```

* We don't use the raw input `x` directly.
* We pass `x` through three different independent Linear layers to create the Q, K, and V vectors.
* **`head_size`**: The size of these new vectors (16). It is often smaller than the original embedding size `C` (32).

#### 3. Calculating Affinity (`wei = q @ k.transpose...`)

This is the "Search" operation.

* **Dot Product:** Mathematically, a dot product measures how aligned two vectors are.
* **The Operation:** We multiply every Query by every Key.
* **High Score:** The Query and Key align. The model found what it was looking for.
* **Low Score:** They are unrelated.


* **Transpose (`-2, -1`)**: We flip the Key matrix so the dimensions align for matrix multiplication: `(B, T, 16) @ (B, 16, T) -> (B, T, T)`.

#### 4. Aggregating Values (`out = wei @ v`)

This is the "Retrieval" operation.

* **Previous Versions:** `wei @ x`. We just averaged the raw inputs.
* **This Version:** `wei @ v`. We aggregate the **Values**.
* **Why?** The information used to *match* (Key) might be different from the information useful to *pass on* (Value).
* *Analogy:* You search Google using "Keywords" (Key), but you read the "Article Content" (Value).



#### 5. Summary of the Flow

1. **Input:** `x` (Raw Token Features).
2. **Process:** Generate Queries, Keys, Values.
3. **Compare:** Match Q and K to get Attention Scores (`wei`).
4. **Filter:** Mask future tokens (`tril`) and Normalize (`softmax`).
5. **Output:** Weighted sum of V (`out`).

---

In [22]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

This specific line of code implements the famous **"Scaled Dot-Product Attention"** mechanism from the *Attention Is All You Need* paper.

It calculates the raw "affinity scores" between tokens while keeping the math stable for the neural network.

Here is the breakdown of the two distinct operations happening in this line:

### 1. The Matrix Multiplication: `q @ k.transpose(-2, -1)`

This calculates the similarity between every Query and every Key.

* **The Shapes:**
* `q`: `(B, T, head_size)`  `(4, 8, 16)`
* `k`: `(B, T, head_size)`  `(4, 8, 16)`


* **The Transpose:**
* We cannot multiply `(4, 8, 16)` by `(4, 8, 16)` directly. Matrix multiplication requires the inner dimensions to match (e.g.,  and ).
* `k.transpose(-2, -1)` flips the last two dimensions (Time and Head Size).
* **New k Shape:** `(B, head_size, T)`  `(4, 16, 8)`.


* **The Result (`q @ k.T`):**
* `(4, 8, 16) @ (4, 16, 8)`  **`(4, 8, 8)`**.
* We recover the `(Batch, Time, Time)` matrix where every position  represents the dot product score between Token  and Token .



---

### 2. The Scaling: `**head_size**-0.5`

This is the **"Scaled"** part. We multiply the scores by .

* **What it is:** `head_size**-0.5` is math notation for .
* If `head_size` is 16, we multiply everything by  (0.25).



#### **Why is this scaling crucial? (The "Softmax Saturation" Problem)**

If you initialize `q` and `k` with standard Gaussian variables (mean 0, variance 1), their dot product can become very large.

**The Math Logic:**

* If you sum up the product of 16 random pairs, the **variance** of the result grows to equal the `head_size` (16).
* This means your raw attention scores could easily range from **-10 to +10** (or higher).

**The Problem with Softmax:**

* Softmax is very sensitive to large numbers.
* `softmax([0.1, 0.2])`  `[0.47, 0.53]` (Diffuse, gradients flow well).
* `softmax([10, 20])`  `[0.000045, 0.999955]` (Sharpened).


* When Softmax output is extremely sharp (one value is 1.0 and the rest are 0.0), the **gradients become zero**. The model essentially "freezes" and stops learning because it thinks it is already 100% confident.

**The Solution:**
By dividing by , we force the variance of the result back to **1.0**. This keeps the numbers small and ensures the Softmax remains "soft" enough for the network to learn effectively.

### Summary

* **`q @ k.T`**: Finds the matches.
* **Scaling**: Keeps the numbers small so the brain doesn't freeze.

### Next Step: Building the `Head` Class

You now have all the components for a single Self-Attention Head.

1. Linear layers for Q, K, V.
2. Attention calculation (this line).
3. Masking (Tril).
4. Softmax.
5. Aggregation (Value).



In [23]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [24]:
k.var()

tensor(1.0449)

In [25]:
q.var()

tensor(1.0700)

In [26]:
wei.var()

tensor(1.0918)

In [27]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [28]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


---

### 📝 Notes: Layer Normalization (`LayerNorm1d`)

This block implements the normalization technique used in GPT. It ensures that the inputs to the deep layers have a consistent range (Mean 0, Variance 1), which prevents the training from becoming unstable.

#### 1. The Key Difference: Rows vs. Columns

The single most important character in this code is the `1` inside the mean/var calculations:

```python
xmean = x.mean(1, keepdim=True)

```

* **Batch Norm (Old way):** Calculates mean down the **Batch** axis (dim 0).
* *Logic:* "What is the average of Feature 5 across all 32 examples?"
* *Problem:* It ties samples together. If your batch size is small (or 1), it breaks.


* **Layer Norm (GPT way):** Calculates mean across the **Feature** axis (dim 1).
* *Logic:* "What is the average of **this specific example** across all its 100 features?"
* *Benefit:* Every row is normalized independently. Row 0 doesn't care about Row 1. This is crucial for text sequences.



#### 2. The Execution Flow

1. **Calculate Statistics:** It grabs the mean and variance for *each* of the 32 samples individually.
2. **Normalize (`xhat`):** It subtracts the mean and divides by the standard deviation.
* **Result:** Every sample is now a "standard gaussian" (centered at 0).


3. **Scale and Shift (`gamma`, `beta`):**
* `self.out = self.gamma * xhat + self.beta`
* **Why?** Maybe the neural network doesn't *want* the data to be 0-centered. It might learn that centering it at 5.0 is better. `gamma` and `beta` are learnable parameters that let the model move the data where it needs to be.



#### 3. Why is `momentum` ignored?

Notice `momentum=0.1` is in `__init__` but never used in `__call__`?

* **Batch Norm** needs momentum to keep a "running average" of stats because it can't see the whole dataset at once.
* **Layer Norm** calculates exact stats on the fly for every single sample. It doesn't need a history or running average.

---



In [29]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [30]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [31]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [32]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Hyperparameters (Configuration) ---
batch_size = 16  # How many independent sequences to process in parallel? (Higher = faster, uses more RAM)
block_size = 32  # Context length: The model looks at up to 32 past characters to predict the next one.
max_iters = 5000 # How many training steps (loops) to run.
eval_interval = 100 # How often to pause training to print the current loss (error).
learning_rate = 1e-3 # The size of the step the optimizer takes. (1e-3 = 0.001).
device = 'cuda' if torch.cuda.is_available() else 'cpu' # Use GPU if available (much faster!), otherwise CPU.
eval_iters = 200 # When calculating loss, average over this many batches to get a smooth number.
n_embd = 64      # Embedding Dimension: The vector size for each token (e.g., 'a' is a vector of 64 numbers).
n_head = 4       # Number of Attention Heads: We split 64 dims into 4 heads of 16 dims each.
n_layer = 4      # Number of Layers: How many Transformer Blocks we stack on top of each other.
dropout = 0.0    # Regularization: Randomly turns off neurons during training to prevent "memorizing" data.
# --------------------------------------

torch.manual_seed(1337) # Fixes the random number generator so you get the same results every time you run this.

# --- Data Preparation ---
# Load the text file
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Build the vocabulary (unique characters)
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Mappings: Convert char to integer (stoi) and integer to char (itos)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # Function to turn string "hi" into list [4, 7]
decode = lambda l: ''.join([itos[i] for i in l]) # Function to turn list [4, 7] back to "hi"

# Create the dataset tensor
data = torch.tensor(encode(text), dtype=torch.long) # Move data into a PyTorch Tensor (array)
n = int(0.9*len(data)) # Split point: 90% for training, 10% for validation
train_data = data[:n]
val_data = data[n:]

# --- Data Loader ---
def get_batch(split):
    # Select the correct dataset
    data = train_data if split == 'train' else val_data

    # Generate random starting positions (indices) for the batch
    # randint(high, size): Picks random integers between 0 and len(data)-block_size
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Stack the context rows (x) and target rows (y)
    # x: The input chunk (e.g., characters 0 to 31)
    # y: The target chunk (e.g., characters 1 to 32) -> predicting the NEXT character
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    # Move data to GPU (if available)
    x, y = x.to(device), y.to(device)
    return x, y

# --- Loss Estimator (Helper Function) ---
@torch.no_grad() # Context manager: Tells PyTorch "Don't calculate gradients here." Saves memory/speed.
def estimate_loss():
    out = {}
    model.eval() # Switch model to 'eval' mode (turns off Dropout, changes BatchNorm behavior)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item() # .item() converts a 1-element tensor to a standard Python float
        out[split] = losses.mean()
    model.train() # Switch model back to 'train' mode (re-enable Dropout, etc.)
    return out

# --- The Self-Attention Head ---
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        # Linear Projections: These layers learn to transform the input into Q, K, V
        self.key = nn.Linear(n_embd, head_size, bias=False)   # What do I contain?
        self.query = nn.Linear(n_embd, head_size, bias=False) # What am I looking for?
        self.value = nn.Linear(n_embd, head_size, bias=False) # What information do I pass along?

        # 'register_buffer': Creates a tensor that is part of the state, but not a trainable parameter (no gradients).
        # We use this for the 'tril' mask (the lower triangular matrix of 1s).
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)

        # --- Compute Attention Scores (Affinities) ---
        # Matrix Multiply Q and K. The transpose(-2, -1) flips the last two dimensions to allow multiplication.
        # * C**-0.5: Scaled Dot-Product. Divides by sqrt(head_size) to keep numbers small (prevents softmax freezing).
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)

        # --- Masking ---
        # masked_fill: Replaces positions where tril == 0 (the future) with -inf (negative infinity).
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)

        # --- Normalization ---
        # softmax: Converts scores into probabilities (0 to 1). -inf becomes 0.
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)

        # --- Aggregation ---
        # Perform weighted sum of the Values based on the attention scores (wei)
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

# --- Multi-Head Attention ---
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        # ModuleList: A list of modules that PyTorch tracks properly (like a Python list but for layers)
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # Projection: A final linear layer to mix the results from all heads together
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run all heads in parallel and concatenate (stick together) their outputs on the last dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# --- Feed Forward Network ---
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        # Sequential: Runs layers one after another
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # Expand: Multiply dimension by 4 (standard Transformer trick)
            nn.ReLU(),                     # Activation: Allows learning non-linear patterns
            nn.Linear(4 * n_embd, n_embd), # Contract: Project back down to original dimension
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# --- Transformer Block ---
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size) # Communication (Self-Attention)
        self.ffwd = FeedFoward(n_embd)                  # Computation (Feed Forward)
        self.ln1 = nn.LayerNorm(n_embd)                 # Normalization 1
        self.ln2 = nn.LayerNorm(n_embd)                 # Normalization 2

    def forward(self, x):
        # Residual Connections (The '+'): x = x + ...
        # This allows gradients to flow directly through the network, preventing them from vanishing.
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# --- The Bigram Model (Actually a Transformer) ---
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # Embedding Layers:
        # 1. Token Embedding: What is the character? (Content)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # 2. Position Embedding: Where is the character? (Position)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # The stack of Transformer Blocks
        # The '*' unpacaks the list of blocks into arguments for Sequential
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

        self.ln_f = nn.LayerNorm(n_embd) # Final normalization layer
        self.lm_head = nn.Linear(n_embd, vocab_size) # Language Model Head: Projects to vocabulary size

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)

        # Combine content and position information
        x = tok_emb + pos_emb # (B,T,C)

        # Pass through the deep network
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            # Reshaping for Cross Entropy
            # PyTorch expects (Batch_Size * Time, Channels) for the input
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # Flatten the batch and time dimensions
            targets = targets.view(B*T)  # Flatten the targets too
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop context: Important! We can only feed in the last 'block_size' tokens
            # because our Positional Embeddings only go up to block_size.
            idx_cond = idx[:, -block_size:]

            # Get the predictions
            logits, loss = self(idx_cond)

            # Focus only on the last time step (the prediction for the next character)
            logits = logits[:, -1, :] # becomes (B, C)

            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # Sample from the distribution (introduce randomness/variety)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

# --- Training ---
model = BigramLanguageModel()
m = model.to(device) # Move entire model to GPU or CPU
# print the number of parameters in the model (e.g., 0.2 Million)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# Create a PyTorch optimizer (AdamW is standard for Transformers)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # Every once in a while, evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Evaluate the loss
    logits, loss = model(xb, yb)

    # --- Standard PyTorch Training Step ---
    optimizer.zero_grad(set_to_none=True) # 1. Clear old gradients
    loss.backward()                       # 2. Calculate new gradients (Backprop)
    optimizer.step()                      # 3. Update parameters

# --- Generation ---
context = torch.zeros((1, 1), dtype=torch.long, device=device) # Start with a single '0' token
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5091, val loss 2.5058
step 300: train loss 2.4197, val loss 2.4336
step 400: train loss 2.3501, val loss 2.3562
step 500: train loss 2.2963, val loss 2.3125
step 600: train loss 2.2407, val loss 2.2496
step 700: train loss 2.2054, val loss 2.2187
step 800: train loss 2.1633, val loss 2.1866
step 900: train loss 2.1241, val loss 2.1504
step 1000: train loss 2.1036, val loss 2.1306
step 1100: train loss 2.0698, val loss 2.1180
step 1200: train loss 2.0380, val loss 2.0791
step 1300: train loss 2.0248, val loss 2.0634
step 1400: train loss 1.9926, val loss 2.0359
step 1500: train loss 1.9697, val loss 2.0287
step 1600: train loss 1.9627, val loss 2.0477
step 1700: train loss 1.9403, val loss 2.0115
step 1800: train loss 1.9090, val loss 1.9941
step 1900: train loss 1.9092, val loss 1.9858
step 2000: train loss 1.8847, val loss 1.9925
step 2100: train loss 1.